# Framework comparativo de metaheurísticas para previsão de malária

Notebook inicial do estudo de previsão mensal de casos autóctones em **Breves, Pará**. A fonte foi adaptada para o CSV enviado:

`6- Autóctones Mun Fx Etária e Sexo_Dados completos_data.csv`

A implementação começa pelas fases do prompt: leitura e inspeção dos dados, análise exploratória, agregação mensal, criação de atributos temporais e um baseline com validação temporal. As células estão sem resultados salvos; as métricas e gráficos só aparecem quando forem executados com os dados.

## Como o CSV será interpretado

- O arquivo é delimitado por ponto e vírgula (`;`) e usa UTF-8 com BOM.
- A filtragem usa **município de infecção** igual a Breves e UF `PA`.
- O arquivo já é identificado como dados de casos autóctones; o notebook não aplica um filtro adicional de categoria.
- A coluna temporal disponível é **Data de Notificação**. Ela será usada como referência mensal; notificação não é necessariamente o mês de início dos sintomas ou da infecção.
- `Número de registros` será somado para obter os casos mensais. Os registros do CSV também permitem explorar espécie, sexo e faixa etária.
- O arquivo não traz população. Portanto, este notebook não calcula incidência nem casos importados.
- O CSV tem cerca de 133 MB; a leitura é feita em blocos e guarda na memória apenas as linhas de Breves-PA.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

CONFIG = {
    "municipio": "Breves",
    "uf": "PA",
    "chunksize": 250_000,
    "test_months": 24,
    "cv_splits": 5,
    "random_seed": 42,
    "n_estimators": 300,
}

CSV_FILENAME = "6- Autóctones Mun Fx Etária e Sexo_Dados completos_data.csv"
CANDIDATE_PATHS = [
    Path.home() / "Downloads" / CSV_FILENAME,
    Path.cwd() / CSV_FILENAME,
    Path.cwd().parent / "Downloads" / CSV_FILENAME,
]
DATA_PATH = next((path for path in CANDIDATE_PATHS if path.is_file()), None)
if DATA_PATH is None:
    searched = "\n".join(str(path) for path in CANDIDATE_PATHS)
    raise FileNotFoundError(
        f"CSV não encontrado. Coloque o arquivo em Downloads ou atualize DATA_PATH.\n{searched}"
    )

print(f"Arquivo selecionado: {DATA_PATH}")

## Fase 1 — carregar e preparar a fonte

O cabeçalho é normalizado removendo espaços laterais, pois o CSV tem um espaço no final do nome da coluna de município. A leitura em blocos evita carregar os 133 MB completos de uma vez.

In [ ]:
# Nomes reconhecidos no CSV após remover BOM e espaços laterais.
REQUIRED_COLUMNS = {
    "BR Mun de infecção": "municipio_infeccao",
    "BR UF Infecção": "uf_infeccao",
    "BR Data de Notificação": "data_notificacao",
    "Número de registros": "numero_registros",
    "BR Espécie Parasitária": "especie",
    "BR Sexo": "sexo",
    "BR Faixa etária": "faixa_etaria",
}

header = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig", nrows=0)
normalized_to_original = {
    str(column).replace("\ufeff", "").strip(): column
    for column in header.columns
}
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(normalized_to_original))
if missing_columns:
    raise ValueError(f"Colunas esperadas não encontradas no CSV: {missing_columns}")

source_columns = [normalized_to_original[name] for name in REQUIRED_COLUMNS]
rename_columns = {
    normalized_to_original[original]: normalized
    for original, normalized in REQUIRED_COLUMNS.items()
}
filtered_chunks = []
rows_read = 0

for chunk in pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8-sig",
    usecols=source_columns,
    dtype={column: "string" for column in source_columns},
    chunksize=CONFIG["chunksize"],
):
    rows_read += len(chunk)
    chunk = chunk.rename(columns=rename_columns)
    municipality = chunk["municipio_infeccao"].fillna("").str.strip().str.lower()
    state = chunk["uf_infeccao"].fillna("").str.strip().str.upper()
    selected = chunk.loc[
        municipality.eq(CONFIG["municipio"].lower()) & state.eq(CONFIG["uf"])
    ].copy()
    if not selected.empty:
        filtered_chunks.append(selected)

if not filtered_chunks:
    raise ValueError("Nenhuma linha foi encontrada para Breves-PA.")

records = pd.concat(filtered_chunks, ignore_index=True)
records["peso"] = pd.to_numeric(records["numero_registros"], errors="coerce")
invalid_weights = int(records["peso"].isna().sum())
records["peso"] = records["peso"].fillna(0)
records["data_notificacao"] = pd.to_datetime(
    records["data_notificacao"], format="%d/%m/%Y", errors="coerce"
)
invalid_dates = int(records["data_notificacao"].isna().sum())
records = records.dropna(subset=["data_notificacao"]).copy()
records["mes"] = records["data_notificacao"].dt.to_period("M").dt.to_timestamp()

print(f"Linhas lidas do CSV: {rows_read:,}")
print(f"Registros de Breves-PA com data válida: {len(records):,}")
print(f"Pesos não numéricos convertidos em zero: {invalid_weights:,}")
print(f"Datas inválidas descartadas: {invalid_dates:,}")
print(f"Período observado: {records['data_notificacao'].min():%d/%m/%Y} a {records['data_notificacao'].max():%d/%m/%Y}")

## Inspeção e análise exploratória

As próximas células resumem a série e mostram distribuição temporal, sazonalidade por mês do calendário, espécies, sexo e faixas etárias. Essas quebras são exploratórias; não são usadas como atributos contemporâneos do modelo inicial.

In [ ]:
monthly_cases = (
    records.groupby("mes")["peso"]
    .sum()
    .sort_index()
    .rename("casos")
)
monthly_index = pd.date_range(
    monthly_cases.index.min(), monthly_cases.index.max(), freq="MS", name="mes_origem"
)
monthly_cases = monthly_cases.reindex(monthly_index, fill_value=0).astype(float)

summary = pd.Series({
    "primeiro mês": monthly_cases.index.min().strftime("%Y-%m"),
    "último mês": monthly_cases.index.max().strftime("%Y-%m"),
    "meses na série": len(monthly_cases),
    "meses sem registro preenchidos com zero": int((monthly_cases == 0).sum()),
    "soma dos registros": int(monthly_cases.sum()),
    "média mensal": round(monthly_cases.mean(), 2),
    "mediana mensal": round(monthly_cases.median(), 2),
    "máximo mensal": int(monthly_cases.max()),
})
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
monthly_cases.plot(ax=axes[0, 0], color="#177e73", linewidth=1.2)
axes[0, 0].set_title("Casos autóctones notificados por mês — Breves-PA")
axes[0, 0].set_ylabel("Registros")

monthly_cases.groupby(monthly_cases.index.month).mean().plot(
    kind="bar", ax=axes[0, 1], color="#7abf9d"
)
axes[0, 1].set_title("Média por mês do calendário")
axes[0, 1].set_xlabel("Mês")
axes[0, 1].set_ylabel("Média de registros")
axes[0, 1].tick_params(axis="x", rotation=0)

for column, title, color in [
    ("especie", "Registros por espécie", "#77a8a1"),
    ("faixa_etaria", "Registros por faixa etária", "#d0a15a"),
]:
    totals = records.assign(**{column: records[column].fillna("Não informado").str.strip()})
    totals = totals.groupby(column)["peso"].sum().sort_values(ascending=False)
    ax = axes[1, 0] if column == "especie" else axes[1, 1]
    totals.plot(kind="bar", ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Registros")
    ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

records.groupby("sexo")["peso"].sum().sort_values(ascending=False).to_frame("registros")

## Fases 2–3 — série mensal e engenharia temporal

Cada linha de modelagem representa o **mês de origem** `t`. O alvo `target_t+1` é o total notificado no mês seguinte. Todos os lags e estatísticas móveis usam somente dados observados até `t`; mês, trimestre e seno/cosseno representam o calendário conhecido do mês previsto.

As colunas `target_t+3` e `target_t+6` ficam disponíveis para etapas futuras, mas não entram no primeiro experimento.

In [ ]:
features = pd.DataFrame(index=monthly_cases.index)
features["casos_mes_atual"] = monthly_cases

for lag in (1, 2, 3, 6, 12):
    features[f"lag_{lag}"] = monthly_cases.shift(lag)

for window in (3, 6, 12):
    features[f"media_{window}m"] = monthly_cases.rolling(
        window=window, min_periods=window
    ).mean()

for window in (3, 6):
    features[f"desvio_{window}m"] = monthly_cases.rolling(
        window=window, min_periods=window
    ).std()

features["tendencia_3m"] = monthly_cases - monthly_cases.shift(3)
month_to_predict = monthly_cases.index + pd.DateOffset(months=1)
features["mes"] = month_to_predict.month
features["ano"] = month_to_predict.year
features["trimestre"] = month_to_predict.quarter
features["sin_mes"] = np.sin(2 * np.pi * features["mes"] / 12)
features["cos_mes"] = np.cos(2 * np.pi * features["mes"] / 12)
features["estacao_periodo"] = pd.Series(
    [f"T{quarter}" for quarter in month_to_predict.quarter],
    index=features.index,
)
features["indice_temporal"] = np.arange(len(features))
features = pd.get_dummies(
    features, columns=["estacao_periodo"], prefix="periodo", dtype=float
)

targets = pd.DataFrame(index=monthly_cases.index)
targets["target_t+1"] = monthly_cases.shift(-1)
targets["target_t+3"] = monthly_cases.shift(-3)
targets["target_t+6"] = monthly_cases.shift(-6)

feature_columns = list(features.columns)
model_data = features.join(targets).dropna(
    subset=feature_columns + ["target_t+1"]
).copy()
print(f"Amostras elegíveis para previsão t+1: {len(model_data)}")
print(f"Número de atributos: {len(feature_columns)}")
model_data[["casos_mes_atual", "lag_1", "lag_12", "media_3m", "target_t+1"]].tail()

### Verificação conceitual contra vazamento temporal

Para prever o mês seguinte, os preditores da linha `t` usam a contagem do próprio mês de origem e meses anteriores. A coluna-alvo é deslocada para trás a partir da série (`shift(-1)`), mas é excluída de `feature_columns`. A divisão e a validação são cronológicas; não há embaralhamento.

## Fase 4 — baseline com validação temporal

O primeiro comparativo usa dois baselines simples (último valor observado e média móvel de três meses) e Random Forest padrão. `TimeSeriesSplit` preserva a ordem temporal. Os 24 meses finais ficam reservados para teste final, configurável em `CONFIG`; a quantidade de meses também pode ser reduzida se a série disponível for menor.

In [ ]:
TEST_MONTHS = CONFIG["test_months"]
N_SPLITS = CONFIG["cv_splits"]
if len(model_data) <= TEST_MONTHS + N_SPLITS + 1:
    raise ValueError(
        "Poucas amostras após os lags para o tamanho de teste e número de folds escolhidos. "
        "Reduza CONFIG['test_months'] ou CONFIG['cv_splits']."
    )

development = model_data.iloc[:-TEST_MONTHS].copy()
test = model_data.iloc[-TEST_MONTHS:].copy()
print(f"Desenvolvimento: {development.index.min():%Y-%m} a {development.index.max():%Y-%m}")
print(f"Teste final: {test.index.min():%Y-%m} a {test.index.max():%Y-%m}")


def calculate_metrics(y_true, y_pred):
    """Calcula métricas; MAPE ignora meses cujo valor real é zero."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    nonzero = y_true != 0
    denominator = np.abs(y_true) + np.abs(y_pred)
    valid_smape = denominator > 0
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAPE_pct_sem_zeros": (
            np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
            if nonzero.any() else np.nan
        ),
        "sMAPE_pct": (
            np.mean(2 * np.abs(y_true[valid_smape] - y_pred[valid_smape]) / denominator[valid_smape]) * 100
            if valid_smape.any() else np.nan
        ),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan,
    }


def make_random_forest(seed=42):
    """Cria o baseline reproduzível; hiperparâmetros ficam fixos nesta fase."""
    return RandomForestRegressor(
        n_estimators=CONFIG["n_estimators"],
        min_samples_leaf=2,
        max_features=0.8,
        random_state=seed,
        n_jobs=-1,
    )

In [ ]:
# Walk-forward folds somente na parte de desenvolvimento.
cv = TimeSeriesSplit(n_splits=N_SPLITS)
cv_rows = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(development), start=1):
    train_fold = development.iloc[train_idx]
    valid_fold = development.iloc[valid_idx]
    y_train = train_fold["target_t+1"]
    y_valid = valid_fold["target_t+1"]

    rf = make_random_forest(CONFIG["random_seed"] + fold)
    rf.fit(train_fold[feature_columns], y_train)
    predictions = {
        "Naive (último mês)": valid_fold["casos_mes_atual"].to_numpy(),
        "Média móvel 3m": valid_fold["media_3m"].to_numpy(),
        "Random Forest": rf.predict(valid_fold[feature_columns]),
    }
    for model_name, y_pred in predictions.items():
        cv_rows.append({
            "fold": fold,
            "modelo": model_name,
            **calculate_metrics(y_valid, y_pred),
        })

cv_results = pd.DataFrame(cv_rows)
cv_summary = cv_results.groupby("modelo").agg(
    RMSE_medio=("RMSE", "mean"),
    RMSE_desvio=("RMSE", "std"),
    MAE_medio=("MAE", "mean"),
    sMAPE_medio=("sMAPE_pct", "mean"),
).sort_values("RMSE_medio")
cv_summary

In [ ]:
# Ajuste no período de desenvolvimento e avaliação única no holdout temporal.
final_rf = make_random_forest(CONFIG["random_seed"])
final_rf.fit(development[feature_columns], development["target_t+1"])
final_predictions = {
    "Naive (último mês)": test["casos_mes_atual"].to_numpy(),
    "Média móvel 3m": test["media_3m"].to_numpy(),
    "Random Forest": final_rf.predict(test[feature_columns]),
}

test_rows = []
for model_name, y_pred in final_predictions.items():
    test_rows.append({"modelo": model_name, **calculate_metrics(test["target_t+1"], y_pred)})
test_results = pd.DataFrame(test_rows).sort_values("RMSE")
test_results

In [ ]:
# Gráfico do holdout e importância das variáveis do Random Forest.
test_target_months = test.index + pd.DateOffset(months=1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(test_target_months, test["target_t+1"], label="Real", color="#222222", linewidth=2)
for name, values in final_predictions.items():
    axes[0].plot(test_target_months, values, label=name, alpha=0.85)
axes[0].set_title("Casos reais e previstos — teste temporal")
axes[0].set_ylabel("Registros")
axes[0].legend()

importance = pd.Series(
    final_rf.feature_importances_, index=feature_columns
).sort_values().tail(15)
importance.plot(kind="barh", ax=axes[1], color="#177e73")
axes[1].set_title("Random Forest — importância de atributos")
axes[1].set_xlabel("Importância")
plt.tight_layout()
plt.show()

## Próximas fases do framework

Este notebook estabelece a fonte CSV e o baseline temporal. As etapas seguintes previstas no prompt são:

1. Implementar função fitness com RMSE e penalização configurável por complexidade e custo.
2. Rodar GA-RF, PSO-RF, DE-RF, GWO-RF e WOA-RF com seeds e execuções independentes.
3. Guardar curvas de convergência, parâmetros, métricas e previsões sem sobrescrever resultados anteriores.
4. Comparar as execuções com Friedman e pós-teste apropriado.
5. Expandir para XGBoost, SVR e MLP apenas após validar o primeiro experimento.

A base fornecida não contém população, então incidência exigirá uma fonte populacional compatível por município e ano. Como a data é de notificação, uma análise de sensibilidade com mês de início dos sintomas seria importante se essa variável estiver disponível em outra fonte. Nenhum ranking ou conclusão científica é pré-preenchido.